In [ ]:
# Install the Cohere Python SDK
#The cohere library is used to connect Python applications to Cohere's AI models.
!pip install -U cohere

In [ ]:
# Import the Cohere Python SDK
import cohere

# Import NumPy for numerical operations
import numpy as np

# Import Pandas for data manipulation and analysis
import pandas as pd

# Import tqdm to display progress bars in loops
from tqdm import tqdm

# Never share your API key publicly
api_key = "cohere_VioMO9X5xhCquAeI64QeL7fdzQyuycSxdM6sGJRl3m0yFq"

# Create a Cohere client to communicate with the Cohere API
co = cohere.Client(api_key)

In [ ]:
# Store a paragraph about the movie Interstellar
text = """
Interstellar is a 2014 epic science fiction film co-written,
directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain,
Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to
survive, the film follows a group of astronauts who travel
through a wormhole near Saturn in search of a new home for
mankind.
Brothers Christopher and Jonathan Nolan wrote the screenplay,
which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in
Physics[4] Kip Thorne was an executive producer, acted as a
scientific consultant, and wrote a tie-in book, The Science of
Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in
the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in
Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and
the company Double Negative created additional digital effects.
Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock,
expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773
million with subsequent re-releases), making it the tenth-highest
grossing film of 2014.
It received acclaim for its performances, direction, screenplay,
musical score, visual effects, ambition, themes, and emotional
weight.
It has also received praise from many astronomers for its
scientific accuracy and portrayal of theoretical astrophysics.
Since its premiere, Interstellar gained a cult following,[5], and
now is regarded by many sci-fi experts as one of the best
science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy
Awards, winning Best Visual Effects, and received numerous other
accolades.
"""

# Split the paragraph into individual sentences using '.' as the separator
texts = text.split('.')

# Remove extra spaces and newline characters from each sentence
texts = [t.strip(' \n') for t in texts]

In [ ]:
# Generate embeddings for the text using Cohere's embedding model
response = co.embed(

    # List of sentences/documents to embed
    texts=texts,

    # Specify the embedding model
    model="embed-v4.0",

    # Specify that the input consists of searchable documents
    input_type="search_document"
)

# Convert the returned embeddings into a NumPy array
embeds = np.array(response.embeddings)

# Print the dimensions of the embedding matrix
print(embeds.shape)

In [ ]:
# Install the CPU version of the FAISS library
# FAISS is a library used to store and search embeddings efficiently.
!pip install faiss-cpu

In [ ]:
# Import the FAISS library
import faiss

# Get the dimension (number of features) of each embedding vector
dim = embeds.shape[1]

# Create a FAISS index using L2 (Euclidean) distance
index = faiss.IndexFlatL2(dim)

# Check whether the index requires training
print(index.is_trained)

# Add all embedding vectors to the FAISS index by converting float 32 bit to float 16
index.add(np.float32(embeds))

In [ ]:
# Define a function to perform semantic search
def search(query, number_of_results=3):

    # Generate an embedding for the user's query
    query_embed = co.embed(
        texts=[query],
        model="embed-v4.0",
        input_type="search_query",
    ).embeddings[0]

    # Search the FAISS index for the most similar embeddings
    distances, similar_item_ids = index.search(
        np.float32([query_embed]),
        number_of_results
    )

    # Convert the list of texts into a NumPy array for indexing
    texts_np = np.array(texts)

    # Create a DataFrame containing the retrieved texts and their distances
    results = pd.DataFrame({
        "texts": texts_np[similar_item_ids[0]],
        "distance": distances[0]
    })

    # Display the query
    print(f"Query: '{query}'\nNearest neighbors:")

    # Return the search results
    return results

In [ ]:
# Define the search query
query = "how precise was the science"

# Perform semantic search using the query
results = search(query)

# Display the retrieved search results
results

In [ ]:
# Install the rank_bm25 library for keyword-based document retrieval
!pip install rank_bm25

In [ ]:
# Import the BM25 ranking algorithm.
# BM25 is used for keyword-based document search.
from rank_bm25 import BM25Okapi

# Import the list of English stop words.
# Stop words are common words like "the", "is", "and", etc.
from sklearn.feature_extraction import _stop_words

# Import the string module to remove punctuation.
import string


# Function to tokenize and clean the input text
def bm25_tokenizer(text):

    # Create an empty list to store the processed words
    tokenized_doc = []

    # Convert the text to lowercase and split it into words
    for token in text.lower().split():

        # Remove punctuation marks from the word
        token = token.strip(string.punctuation)

        # Keep the word only if:
        # 1. It is not empty
        # 2. It is not a stop word
        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)

    # Return the cleaned list of tokens
    return tokenized_doc


# Tokenize every document in the dataset
tokenized_corpus = []
# Loop through each document (passage) in the text collection
for passage in tqdm(texts):

    # Tokenize the document and add the tokens to the BM25 corpus
    tokenized_corpus.append(bm25_tokenizer(passage))

# Build the BM25 search index
bm25 = BM25Okapi(tokenized_corpus)


# Function to perform keyword-based search
def keyword_search(query, top_k=3, num_candidates=15):

    # Print the user's search query
    print("Input question:", query)

    # BM25 keyword search

    # Calculate a BM25 score for every document
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))

    # Get the indices of the top candidate documents
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]

    # Store the document index and its BM25 score
    bm25_hits = [
        {"corpus_id": idx, "score": bm25_scores[idx]}
        for idx in top_n
    ]

    # Sort the documents from highest score to lowest
    bm25_hits = sorted(
        bm25_hits,
        key=lambda x: x["score"],
        reverse=True
    )

    # Display the top matching documents
    print(f"Top-{top_k} lexical search (BM25) hits")

    for hit in bm25_hits[:top_k]:
        print(
            "\t{:.3f}\t{}".format(
                hit["score"],
                texts[hit["corpus_id"]].replace("\n", " ")
            )
        )

In [ ]:
# Perform a keyword-based search using the BM25 algorithm
keyword_search(query="how precise was the science")

In [ ]:
# Define the search query
query = "how precise was the science"

# Rerank all documents using Cohere's reranking model
results = co.rerank(
    query=query,
    documents=texts,
    top_n=3,
    return_documents=True
)

# Display the top 3 reranked results
results.results

In [ ]:
# Loop through each reranked result with its index
for idx, result in enumerate(results.results):

    # Print the rank, relevance score, and document text
    print(idx, result.relevance_score, result.document.text)

In [ ]:
# Define a function that combines BM25 keyword search with Cohere reranking
def keyword_and_reranking_search(query, top_k=3, num_candidates=10):

    # Print the user's query
    print("Input question:", query)

    # Compute BM25 scores for all documents
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))

    # Get the indices of the top BM25 candidate documents
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]

    # Store the candidate document IDs and their BM25 scores
    bm25_hits = [
        {"corpus_id": idx, "score": bm25_scores[idx]}
        for idx in top_n
    ]

    # Sort the candidates by BM25 score in descending order
    bm25_hits = sorted(
        bm25_hits,
        key=lambda x: x["score"],
        reverse=True
    )

    # Display the top BM25 search results
    print(f"Top-{top_k} lexical search (BM25) hits")

    # Print the top BM25 results
    for hit in bm25_hits[:top_k]:
        print(
            "\t{:.3f}\t{}".format(
                hit["score"],
                texts[hit["corpus_id"]].replace("\n", " ")
            )
        )

    # Collect the BM25 candidate documents for reranking
    docs = [texts[hit["corpus_id"]] for hit in bm25_hits]

    # Display the reranking heading
    print(f"\n\nTop-{top_k} hits by rank-API ({len(bm25_hits)} BM25 hits re-ranked)")

    # Rerank the BM25 candidate documents using Cohere
    results = co.rerank(
        query=query,
        documents=docs,
        top_n=top_k,
        return_documents=True
    )

    # Print the reranked results
    for hit in results.results:
        print(
            "\t{:.3f}\t{}".format(
                hit.relevance_score,
                hit.document.text.replace("\n", " ")
            )
        )

In [ ]:
keyword_and_reranking_search(query = "how precise was the science")

In [ ]:
# Define the user's query
query = "income generated"

#  Retrieve the most relevant documents
# Here we use embedding-based search (semantic search)
# In practice, a hybrid search (BM25 + embeddings) is often preferred
results = search(query)

# Convert the retrieved documents into the format expected by Cohere Chat
docs_dict = [
    {"text": text}
    for text in results["texts"]
]

# Step 3: Generate an answer using the retrieved documents as context
response = co.chat(
    message=query,
    documents=docs_dict
)

# Display the generated answer
print(response.text)

In [ ]:
# Download the Phi-3 Mini 4K Instruct GGUF model from Hugging Face
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

In [ ]:
# Uninstall any existing LangChain packages to avoid version conflicts
!pip uninstall -y langchain langchain-core langchain-community

# Install the specific LangChain version
!pip install langchain==0.0.353

# Install the llama-cpp-python library to run GGUF models locally
!pip install llama-cpp-python

In [ ]:
# Import the LlamaCpp wrapper from LangChain
from langchain import LlamaCpp

# Load the local GGUF language model
llm = LlamaCpp(

    # Path to the GGUF model file
    model_path="Phi-3-mini-4k-instruct-fp16.gguf",

    # Number of model layers to run on the GPU
    # Use -1 to offload all layers to the GPU (if a GPU is available)
    n_gpu_layers=-1,

    # Maximum number of tokens the model can generate
    max_tokens=500,

    # Maximum context window (input + output tokens)
    n_ctx=2048,

    # Random seed for reproducible outputs
    seed=42,

    # Disable detailed execution logs
    verbose=False
)

In [ ]:
# Import the Hugging Face embedding model class
from langchain.embeddings.huggingface import HuggingFaceEmbeddings

# Load the embedding model for converting text into vector embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="thenlper/gte-small"
)

# Import the FAISS vector database from LangChain
from langchain.vectorstores import FAISS

# Create a FAISS vector database from the text documents using the embedding model
db = FAISS.from_texts(texts, embedding_model)

In [ ]:
# Import PromptTemplate for creating custom prompts
# A PromptTemplate lets us create a reusable prompt with placeholders.
from langchain import PromptTemplate

# Create a prompt template for Retrieval-Augmented Generation (RAG)
template = """<|user|>
Relevant information:
{context}

Provide a concise answer to the following question using the relevant information provided above:

{question}
<|end|>
<|assistant|>"""

# Define the prompt and specify the variables it expects
prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

# Import the RetrievalQA chain
#This class automatically combines Retriever Prompt LLM into one pipeline.
from langchain.chains import RetrievalQA

# Create the Retrieval-Augmented Generation (RAG) pipeline
rag = RetrievalQA.from_chain_type(

    # Language model used to generate the answer
    llm=llm,

    # Combine all retrieved documents into one prompt
    chain_type="stuff",

    # Use the FAISS vector database as the retriever
    retriever=db.as_retriever(),

    # Pass the custom prompt to the QA chain
    chain_type_kwargs={
        "prompt": prompt
    },

    # Display intermediate execution details
    verbose=True
)

In [ ]:
# Ask the RetrievalQA pipeline a question
rag.invoke("Income generated")